# CNN Comparison

Use this notebook after running `python src/cnn.py`. It compares the CNN training history and final metrics against the HOG + SVM baseline.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"

HISTORY_PATH = DATA_DIR / "cnn_history.json"
RESULTS_PATH = DATA_DIR / "cnn_results.json"
TEST_PREDS_PATH = DATA_DIR / "cnn_test_predictions.npz"

SVM_TEST_TOP1 = 0.0910
SVM_TEST_TOP5 = 0.2171

plt.style.use("default")

## Training Curves

In [ ]:
if not HISTORY_PATH.exists():
    raise FileNotFoundError(f"Run src/cnn.py first. Missing: {HISTORY_PATH}")

history = json.loads(HISTORY_PATH.read_text())
history

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

epochs = [row["epoch"] for row in history]

axes[0].plot(epochs, [row["train_loss"] for row in history], marker="o", label="Train")
axes[0].plot(epochs, [row["val_loss"] for row in history], marker="o", label="Val")
axes[0].set_title("CNN Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].legend()

axes[1].plot(epochs, [row["train_top1"] for row in history], marker="o", label="Train Top-1")
axes[1].plot(epochs, [row["val_top1"] for row in history], marker="o", label="Val Top-1")
axes[1].plot(epochs, [row["train_top5"] for row in history], marker="o", label="Train Top-5")
axes[1].plot(epochs, [row["val_top5"] for row in history], marker="o", label="Val Top-5")
axes[1].set_title("CNN Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1)
axes[1].legend()

plt.tight_layout()

## SVM vs CNN

In [ ]:
results = json.loads(RESULTS_PATH.read_text()) if RESULTS_PATH.exists() else {}
best_val = max(history, key=lambda row: row["val_top1"])

cnn_label = "CNN Val"
cnn_top1 = float(best_val["val_top1"])
cnn_top5 = float(best_val["val_top5"])

if "test" in results:
    cnn_label = "CNN Test"
    cnn_top1 = results["test"]["top1"]
    cnn_top5 = results["test"]["top5"]

metrics = [
    {"model": "HOG + SVM Test", "top1": SVM_TEST_TOP1, "top5": SVM_TEST_TOP5},
    {"model": cnn_label, "top1": cnn_top1, "top5": cnn_top5},
]
metrics

In [ ]:
x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - width / 2, [row["top1"] for row in metrics], width, label="Top-1")
ax.bar(x + width / 2, [row["top5"] for row in metrics], width, label="Top-5")
ax.set_xticks(x)
ax.set_xticklabels([row["model"] for row in metrics])
ax.set_ylim(0, 1)
ax.set_ylabel("Accuracy")
ax.set_title("Model Accuracy Comparison")
ax.legend()

for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=3)

plt.tight_layout()

## CNN Test Diagnostics

Run `python src/cnn.py --eval-test-only` or train with `python src/cnn.py --eval-test` before using this section.

In [ ]:
if TEST_PREDS_PATH.exists():
    preds = np.load(TEST_PREDS_PATH)
    y_true = preds["y_true"]
    y_pred = preds["y_pred"]

    classes = np.unique(y_true)
    per_class_acc = np.array([(y_pred[y_true == c] == c).mean() for c in classes])
    sort_idx = np.argsort(per_class_acc)

    fig, ax = plt.subplots(figsize=(18, 5))
    ax.bar(np.arange(len(classes)), per_class_acc[sort_idx])
    ax.axhline(per_class_acc.mean(), color="red", linestyle="--", label=f"Mean: {per_class_acc.mean():.3f}")
    ax.set_title("CNN Per-Class Top-1 Accuracy")
    ax.set_xlabel("Class index, sorted by accuracy")
    ax.set_ylabel("Top-1 accuracy")
    ax.legend()
    plt.tight_layout()
else:
    print(f"No CNN test predictions found at {TEST_PREDS_PATH}")